# Landsat 8/9 STAC Processing Workflow

This notebook drives **`process_landsat_odr`**, the STAC/COG Landsat pipeline. It queries [Earth Search](https://earth-search.aws.element84.com/v1) (collection `landsat-c2-l2`) for Collection 2 Level-2 surface-reflectance scenes that intersect a bounding box, and reads their bands straight from `s3://usgs-landsat`. Nothing is downloaded by hand.

> **This is the current Landsat workflow.** The older `process_landsat89` pipeline, which processes `.tar` archives downloaded from USGS, is retired to [`legacy/landsat_workflow.ipynb`](legacy/landsat_workflow.ipynb). It still runs, and it is the only route to the panchromatic, water-extent and cloud-mask products, which this pipeline does not make.

> **AWS credentials are required.** `s3://usgs-landsat` is a requester-pays bucket: every band read is signed and billed to the caller. On the Disasters hub the pod's own credentials are used and nothing needs configuring.

## Workflow Steps
1. **Configure** the activation: event, area of interest, dates, products.
2. **Preview** the area of interest against the scenes it matches.
3. **Process** each selected product.
4. **View** and plot the results.
5. **Upload** to S3 (opt-in, off by default).

## Products
True color, natural color, color infrared, shortwave infrared, NDVI, NDWI, MNDWI, EVI and NBR. Each can be cloud-masked and mosaicked per satellite pass.

## 1. Configure the activation

Edit this cell for each new activation. Everything below it is plumbing.

In [ ]:
import os
import sys

# Lets a local checkout that has not been pip-installed find the packages.
sys.path.append("../src")

# ==============================================================================
# ACTIVATION OPTIONS BLOCK
# Change these variables for each new disaster activation.
# ==============================================================================

# ------------------------------------------------------------------------------
# Metadata
# ------------------------------------------------------------------------------

# MUST be YYYYMM_Hazard_Location. shared_utils.cog_metadata.resolve_metadata
# splits this into the YEAR_MONTH / HAZARD / LOCATION GeoTIFF tags, so a
# free-text name silently produces incomplete metadata on every product.
EVENT_NAME = "202601_Flood_ExampleCity"

# Landsat is a USGS / NASA mission. Earth Search only redistributes it on AWS,
# so the provenance is still USGS.
SOURCE = "USGS"

# ------------------------------------------------------------------------------
# Landsat data selection
# ------------------------------------------------------------------------------

# Inclusive, YYYY-MM-DD. Landsat 8 and Landsat 9 each revisit every 16 days,
# eight days apart, so a window shorter than about 8 days can match no scene.
START_DATE = "2026-10-09"
END_DATE = "2026-10-12"

# [MIN_LON, MIN_LAT, MAX_LON, MAX_LAT] in WGS84.
# Every scene intersecting this box is processed IN FULL (about 185 km across),
# so a small box is fine -- it selects scenes, it does not crop them.
BBOX = [-87.00, 35.40, -86.80, 35.60]

# Only scenes BELOW this scene-level cloud percentage are processed.
CLOUD_COVER = 50

# ------------------------------------------------------------------------------
# Products -- the names process_landsat_odr --product accepts
# ------------------------------------------------------------------------------

PRODUCTS = {
    "true_color": True,
    "natural_color": False,
    "color_infrared": False,
    "swir": False,
    "ndvi": False,
    "ndwi": False,
    "mndwi": False,
    "evi": False,
    "nbr": False,
}

# ------------------------------------------------------------------------------
# Processing toggles
# ------------------------------------------------------------------------------

# Mask cloud, cloud shadow, cirrus and dilated cloud using the QA_PIXEL band
# that ships with every scene. Indices get the masked pixels as nodata; colour
# composites show them as opaque gray.
ENABLE_MASK = True

# Mosaic the scenes of each satellite pass into one product per pass, instead
# of one file per scene. A date range covering several passes gives several
# mosaics -- passes are never mixed.
ENABLE_MERGE = True

# COG settings are fixed in the CLI and are not options here: native UTM
# projection, ZSTD level 9, nodata -9999 for indices and an alpha band (no
# nodata value) for colour composites.

# ------------------------------------------------------------------------------
# Output
# ------------------------------------------------------------------------------

# A directory of this notebook's own, EMPTIED at the start of every processing
# run. /tmp/s3_temp is shared with other workflows: they would wipe these
# products, and this notebook would wipe theirs.
OUTPUT_DIR = "/tmp/landsat_odr_output"

# ------------------------------------------------------------------------------
# S3 upload -- OFF by default.
#
# Deliberately off: this publishes to the production bucket. Turn it on only
# once you have reviewed the outputs below AND set a real EVENT_NAME.
# ------------------------------------------------------------------------------

ENABLE_S3_UPLOAD = False

# The bucket and every product directory are hard-coded once, in
# shared_utils.product_paths; nothing here keeps its own copy. Products publish
# to s3://<bucket>/ProgramData/Landsat/<Product>/<filename>.tif -- no event and
# no date level, because the activation lives in the GeoTIFF tags.
from shared_utils.product_paths import STAGING_BUCKET

S3_BUCKET = STAGING_BUCKET
PRODUCT_SENSOR = "landsat"   # key into shared_utils.product_paths

In [ ]:
# ==============================================================================
# VALIDATION + ACTIVATION METADATA (auto-populated -- do not edit)
# ==============================================================================
import json
import os
import re
import tempfile

from shared_utils import PROCESSOR_STRING

# --- fail loudly on the mistakes that are expensive to notice later ---

if not re.match(r"^\d{6}_[A-Za-z]+_.+$", EVENT_NAME):
    raise ValueError(
        f"EVENT_NAME {EVENT_NAME!r} must be YYYYMM_Hazard_Location "
        f"(e.g. 202601_Flood_TX). resolve_metadata parses it into the "
        f"YEAR_MONTH / HAZARD / LOCATION tags, so anything else ships "
        f"products with incomplete metadata."
    )

if EVENT_NAME == "202601_Flood_ExampleCity":
    raise ValueError(
        "EVENT_NAME is still the template placeholder. Set a real "
        "activation event before processing."
    )

if not any(PRODUCTS.values()):
    raise ValueError("No products selected -- enable at least one in PRODUCTS.")

if len(BBOX) != 4 or BBOX[0] >= BBOX[2] or BBOX[1] >= BBOX[3]:
    raise ValueError(
        f"BBOX {BBOX} must be [MIN_LON, MIN_LAT, MAX_LON, MAX_LAT] with "
        f"min < max. A transposed box silently returns zero scenes."
    )

# --- metadata embedded as GeoTIFF tags on every output COG ---

ACTIVATION_METADATA = {
    "ACTIVATION_EVENT": EVENT_NAME,
    "SOURCE": SOURCE,
    "PROCESSOR": PROCESSOR_STRING,
}

# One fixed file per workflow, overwritten on every run. A random temp name
# left a new activation_meta_*.json in /tmp each time this cell ran.
ACTIVATION_METADATA_PATH = os.path.join(
    tempfile.gettempdir(), "activation_meta_landsat_odr.json"
)
with open(ACTIVATION_METADATA_PATH, "w") as _f:
    json.dump(ACTIVATION_METADATA, _f, indent=2)

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Activation metadata:")
for key, value in ACTIVATION_METADATA.items():
    print(f"  {key}: {value}")
print(f"\nMetadata file: {ACTIVATION_METADATA_PATH}")
print(f"Output dir:    {OUTPUT_DIR}")
print(f"Date range:    {START_DATE} to {END_DATE}")
print(f"Bounding box:  {BBOX}")
print(f"Products:      {[p for p, on in PRODUCTS.items() if on]}")

## 2. Preview the area of interest

Optional. Plots the bounding box against the Landsat scenes it intersects, so
you can see how much imagery a run will pull.

In [ ]:
# ==============================================================================
# PREVIEW THE AOI
# ==============================================================================
# Each matched Landsat scene can cover a large area (~185 km across).
# The red box shows the AOI; blue boxes show matching Landsat scene footprints.
# This is a preview only and does not run the processing workflow.

try:
    import matplotlib.pyplot as plt
    from pystac_client import Client

    catalog = Client.open("https://earth-search.aws.element84.com/v1")
    collection = "landsat-c2-l2"

    search = catalog.search(
        collections=[collection],
        datetime=f"{START_DATE}/{END_DATE}",
        bbox=BBOX,
        # The same filters process_landsat_odr applies: the collection also
        # holds Landsat 4, 5 and 7, which this workflow does not process.
        query={
            "eo:cloud_cover": {"lt": CLOUD_COVER},
            "platform": {"in": ["landsat-8", "landsat-9"]},
        },
        max_items=50,
    )

    items = list(search.items())

    print(f"{len(items)} Landsat scene(s) match:")

    for item in items:
        cloud = item.properties.get("eo:cloud_cover")
        cloud_text = f"{cloud:.1f}%" if cloud is not None else "unknown"
        print(f"  {item.id}  cloud={cloud_text}")

    fig, ax = plt.subplots(figsize=(8, 8))

    for item in items:
        b = item.bbox

        ax.add_patch(
            plt.Rectangle(
                (b[0], b[1]),
                b[2] - b[0],
                b[3] - b[1],
                fill=False,
                edgecolor="tab:blue",
                linewidth=1,
            )
        )

        # Landsat IDs look like LC09_L2SP_018040_20240923_02_T1.
        # Show the satellite and WRS-2 path/row identifier.
        parts = item.id.split("_")
        label = "_".join(parts[:3]) if len(parts) >= 3 else item.id

        ax.text(
            b[0], b[3], label,
            fontsize=6,
            color="tab:blue",
        )

    # Draw the AOI on top of the scene footprints.
    ax.add_patch(
        plt.Rectangle(
            (BBOX[0], BBOX[1]),
            BBOX[2] - BBOX[0],
            BBOX[3] - BBOX[1],
            fill=False,
            edgecolor="red",
            linewidth=2,
            label="AOI",
        )
    )

    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_title("AOI and matching Landsat scene footprints")
    ax.legend()
    ax.autoscale_view()
    ax.set_aspect("auto")
    plt.tight_layout()
    plt.show()

except ImportError as exc:
    print(f"Preview skipped ({exc}). Processing does not need this cell.")


## 3. Process Landsat Data

One `process_landsat_odr` call per enabled product: the CLI takes a single
`--product` per invocation. Output streams into the cell as it runs. A product
that fails is reported and the remaining products still run; the cell raises at
the end if any failed.

In [ ]:
# ==============================================================================
# PROCESS SELECTED PRODUCTS
# ==============================================================================
import glob
import subprocess

# Start from an EMPTY OUTPUT_DIR, so the view, plot and upload cells below see
# only this run's products. Without it every earlier run's COGs pile up in
# /tmp, and re-running for a new event would publish the old event's files
# again. reset_output_dir refuses any path outside a temp directory, so a real
# folder is never wiped.
from shared_utils import reset_output_dir
reset_output_dir(OUTPUT_DIR)


def _output_snapshot():
    """{path: mtime_ns} for the top-level GeoTIFFs in OUTPUT_DIR."""
    return {
        f: os.stat(f).st_mtime_ns
        for f in glob.glob(os.path.join(OUTPUT_DIR, "*.tif"))
    }


# What the view, plot and upload cells work on: the files of every product
# that FINISHED. OUTPUT_DIR is snapshotted around each product's CLI call so
# that whatever a failed product left behind can be told apart.
NEW_COGS = []
failures = []

for product, enabled in PRODUCTS.items():

    if not enabled:
        continue

    print()
    print("=" * 70)
    print(f"PROCESSING: {product}")
    print("=" * 70)

    process_cmd = [
        "process_landsat_odr",
        "--product", product,
        "--start-date", START_DATE,
        "--end-date", END_DATE,
        "--bbox", *[str(v) for v in BBOX],
        "--cloud-cover", str(CLOUD_COVER),
        "--output", OUTPUT_DIR,
        "--metadata-json", ACTIVATION_METADATA_PATH,
    ]

    if ENABLE_MASK:
        process_cmd.append("--cloud-mask")

    if ENABLE_MERGE:
        process_cmd.append("--merge")

    print(f"Command: {' '.join(process_cmd)}\n")

    _before = _output_snapshot()

    process = subprocess.Popen(
        process_cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        universal_newlines=True,
        bufsize=1,
    )
    for line in process.stdout:
        print(line, end="")

    return_code = process.wait()

    _written = [
        f for f, mtime in _output_snapshot().items() if _before.get(f) != mtime
    ]

    if return_code == 0:
        NEW_COGS += _written
        print(f"\n[ok] {product} completed")
    else:
        # The CLI deletes what it had not finished when it fails, but a
        # process that is killed outright (out of memory) cannot. Whatever a
        # failed product left stays out of NEW_COGS.
        failures.append((product, return_code))
        print(
            f"\n[FAILED] {product} exited {return_code}; "
            f"{len(_written)} unfinished file(s) left out of NEW_COGS"
        )

NEW_COGS = sorted(set(NEW_COGS))

print()
print("=" * 70)
print(f"{len(NEW_COGS)} COG(s) produced by this run:")
for f in NEW_COGS:
    print(f"  {os.path.basename(f)}")

if failures:
    raise RuntimeError(
        f"{len(failures)} Landsat product(s) FAILED: {failures}. "
        f"Review the output above."
    )

## 4. View Results

**Check these before uploading.** A COG that opens is not the same as a COG
that is correct.

In [ ]:
# ==============================================================================
# REVIEW THE COGs THIS RUN PRODUCED
# ==============================================================================
import rasterio as rio

for path in NEW_COGS:
    with rio.open(path) as src:
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f"{os.path.basename(path)}  ({size_mb:.1f} MB)")
        print(
            f"   {src.count} band(s)  {src.dtypes[0]}  nodata={src.nodata}  "
            f"{src.width}x{src.height}  {src.crs}"
        )
        tags = src.tags()
        missing = [
            k for k in ("ACTIVATION_EVENT", "SOURCE", "PROCESSOR")
            if not tags.get(k)
        ]
        if missing:
            print(f"   [WARN] missing metadata tags: {missing}")
        else:
            print(f"   event={tags['ACTIVATION_EVENT']}  source={tags['SOURCE']}")
        print()

if not NEW_COGS:
    print("No COGs produced by this run.")

## Output File Naming Convention

```
Per scene:  <SAT>_<LEVEL>_<product>_<PATHROW>[_masked]_<YYYY-MM-DDTHH:MM:SSZ>.tif
Merged:     <SAT>_<LEVEL>_<product>_P<PATH>_merged[_masked]_<YYYY-MM-DD>_day.tif
```

e.g. `LC08_L2SP_trueColor_021035_masked_2026-10-12T16:24:22Z.tif` (one scene) and
`LC08_L2SP_trueColor_P021_merged_masked_2026-10-12_day.tif` (one pass, mosaicked).

- `<PATHROW>` is the WRS-2 path and row of the scene (`021035`). A merged file
  spans several rows, so it carries only the path (`P021`).
- A merged file is ONE satellite pass. A date range gives one merged file per
  pass, never a mosaic across dates, so it ends in the pass date and `_day`
  instead of a single scene time.
- `_masked` is present when `ENABLE_MASK` is on.

**The activation event is NOT in the filename.** It lives in the GeoTIFF tags
and the S3 prefix. (The legacy `process_landsat89` pipeline prefixes the event
onto filenames instead, so names differ between the two.)

## Product Descriptions

| `PRODUCTS` key | Bands | Description | Use case |
|---|---|---|---|
| `true_color` | red, green, blue | True color | Visual interpretation |
| `natural_color` | swir16, nir08, red | Natural color | Land / water / vegetation contrast |
| `color_infrared` | nir08, red, green | Color infrared | Vegetation health |
| `swir` | swir22, nir08, red | Shortwave infrared | Burn scars, moisture |
| `ndvi` | nir08, red | Normalized Difference Vegetation Index | Vegetation density |
| `ndwi` | green, nir08 | Normalized Difference Water Index | Water detection |
| `mndwi` | green, swir16 | Modified NDWI | Enhanced water detection |
| `evi` | nir08, red, blue | Enhanced Vegetation Index | Vegetation monitoring in dense canopy |
| `nbr` | nir08, swir22 | Normalized Burn Ratio | Burn severity assessment |

Composites are 8-bit RGB with an alpha band. Indices are float32 with nodata `-9999`.

## Next Steps

You can now:
1. Review the plots below, then publish with the upload cell at the end
2. Process additional dates or areas by changing the configuration cell
3. Generate additional products by enabling them in `PRODUCTS`

## 5. Plot Results

Render a quick preview of the products this run generated, to sanity-check the
output before uploading.

In [ ]:
# ==============================================================================
# PREVIEW THE LANDSAT COGs
# ==============================================================================
from shared_utils.plotting import preview_cogs

if NEW_COGS:
    print(f"Previewing {len(NEW_COGS)} Landsat COG(s)...")
    preview_cogs(NEW_COGS, sample_n=10)
else:
    print("No COGs produced by this run.")

## 6. Upload to S3 (Optional)

Publish the finished COGs. **Opt-in**: runs only when `ENABLE_S3_UPLOAD = True` in the configuration cell.

Uploads go to `s3://{S3_BUCKET}/ProgramData/Landsat/<Product>/<filename>`. The product folder is resolved from the filename's product token through `shared_utils.product_paths`, the same table every other workflow and the DPS jobs use.

The upload uses **ambient AWS credentials**: on the hub the pod already assumes `disasters-prod`. Only the COGs produced by the processing cell above are published, and when merging, only the merged COGs.

In [ ]:
# ==============================================================================
# UPLOAD TO S3 (optional)
# ==============================================================================
# Publishes to s3://{S3_BUCKET}/ProgramData/Landsat/<Product>/<filename>
# with AMBIENT AWS credentials (on the hub the pod already assumes
# disasters-prod). Both product_paths helpers must be imported HERE: the
# shared_utils package re-exports upload_file_to_s3 but not these two.
from shared_utils import upload_file_to_s3
from shared_utils.product_paths import prefix_for_product_dir, product_dir

# Filename product token -> the product key in shared_utils.product_paths. Keys
# must match the camelCase token process_landsat_odr writes into the filename,
# and every product in PRODUCTS needs an entry or its COGs cannot be published.
# The directory itself is NOT written here: product_dir() resolves it, so this
# notebook cannot drift from the published layout.
_PRODUCT_KEYS = {
    "trueColor": "trueColor",
    "naturalColor": "naturalColor",
    "colorInfrared": "colorInfrared",
    "swir": "shortwaveInfrared",
    "ndvi": "NDVI",
    "ndwi": "NDWI",
    "mndwi": "MNDWI",
    "nbr": "NBR",
    "evi": "EVI",
}
PRODUCT_FOLDERS = {
    token: product_dir(PRODUCT_SENSOR, key) for token, key in _PRODUCT_KEYS.items()
}

_unmapped = {
    p for p, on in PRODUCTS.items() if on
} - {
    "true_color", "natural_color", "color_infrared", "swir", "ndvi",
    "ndwi", "mndwi", "nbr", "evi",
}
if _unmapped:
    raise ValueError(f"products with no folder mapping: {sorted(_unmapped)}")

if not ENABLE_S3_UPLOAD:
    print(
        "S3 upload OFF. Review the outputs above, then set "
        "ENABLE_S3_UPLOAD = True in the config cell to publish."
    )

else:
    # Scoped to THIS run (NEW_COGS): never a glob of OUTPUT_DIR, which would
    # publish whatever an earlier run left behind.
    cogs = [f for f in NEW_COGS if not f.endswith(".tmp.tif")]

    if ENABLE_MERGE:
        cogs = [f for f in cogs if "merged" in os.path.basename(f)]

    uploaded, skipped = 0, []

    for f in cogs:
        filename = os.path.basename(f)

        subfolder = None
        # Longest token first, so "colorInfrared" is not shadowed by a
        # shorter key that happens to be a substring.
        for token in sorted(PRODUCT_FOLDERS, key=len, reverse=True):
            if f"_{token}_" in filename:
                subfolder = PRODUCT_FOLDERS[token]
                break

        if subfolder is None:
            skipped.append(filename)
            continue

        upload_file_to_s3(
            f, f"s3://{S3_BUCKET}/{prefix_for_product_dir(PRODUCT_SENSOR, subfolder)}/{filename}"
        )
        uploaded += 1

    print(f"\nUploaded {uploaded} COG(s) to "
          f"s3://{S3_BUCKET}/ProgramData/Landsat/<Product>/")
    if skipped:
        raise RuntimeError(
            f"{len(skipped)} COG(s) had no product folder and were NOT "
            f"uploaded: {skipped}. Add the token to _PRODUCT_KEYS -- a "
            f"silent skip is how a whole product goes missing."
        )